# 🎓 UniPulse: Exploratory Data Analysis (EDA) & Summary Aggregation Views
**Data Engine & Star Schema Deliverable**

This notebook performs comprehensive Exploratory Data Analysis (EDA) on the UniPulse Data Warehouse summary aggregation views:
1. `student_analytics`: Individual performance, score standard deviation, pass rates, attendance correlation, health scores.
2. `module_analytics`: Subject difficulty ranking, score variance, min/max metrics, pass rates, attendance vs score correlation.
3. `semester_analytics`: Macro cohort trends, temporal stability, critical risk ratios.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styling & typography
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["figure.figsize"] = (12, 6)

# Add src to sys.path for importing summary_aggregator
sys.path.append(os.path.abspath("../src"))
from summary_aggregator import SummaryAggregator

aggregator = SummaryAggregator()
print("✓ Imports loaded successfully & SummaryAggregator initialized.")

## 1. Load Summary Aggregation Views
Extract student, module, and semester summary analytics tables from the data warehouse.

In [ ]:
student_df = aggregator.fetch_student_analytics()
module_df = aggregator.fetch_module_analytics()
semester_df = aggregator.fetch_semester_analytics()

print(f"Student Analytics Shape : {student_df.shape}")
print(f"Module Analytics Shape  : {module_df.shape}")
print(f"Semester Analytics Shape: {semester_df.shape}")
student_df.head(5)

## 2. Student-Level EDA: GPA & Score StdDev Distribution
Analyze score distributions, standard deviations, and identify student performance variance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distribution
sns.histplot(student_df['mean_assessment_score'], kde=True, ax=axes[0], color='#2563eb', bins=15)
axes[0].set_title('Distribution of Student Mean Assessment Scores')
axes[0].set_xlabel('Mean Score (%)')
axes[0].set_ylabel('Student Count')

# Attendance distribution
sns.histplot(student_df['mean_attendance_rate'], kde=True, ax=axes[1], color='#059669', bins=15)
axes[1].set_title('Distribution of Student Attendance Rates')
axes[1].set_xlabel('Attendance Rate (%)')
axes[1].set_ylabel('Student Count')

plt.tight_layout()
plt.show()

## 3. Module-Level EDA: Pass Rates & Score Variance
Evaluate pass rate percentages, score standard deviation (`stddev_score`), and difficulty across subjects.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

bars = sns.barplot(
    data=module_df.sort_values(by='pass_rate', ascending=False),
    x='module_code',
    y='pass_rate',
    hue='module_code',
    palette='Blues_r',
    legend=False,
    ax=ax
)

ax.set_title('Module Pass Rates (%) Comparison')
ax.set_xlabel('Module Code')
ax.set_ylabel('Pass Rate (%)')
ax.set_ylim(0, 105)

for bar in bars.patches:
    if bar.get_height() > 0:
        ax.annotate(f"{bar.get_height():.1f}%",
                    (bar.get_x() + bar.get_width() / 2., bar.get_height()),
                    ha='center', va='center', xytext=(0, 6), textcoords='offset points')

plt.tight_layout()
plt.show()

## 4. Attendance vs. Performance Correlation Scatter Analysis
Compute and visualize Pearson correlation ($r$) between attendance rates and academic assessment scores.

In [ ]:
plt.figure(figsize=(10, 6))

sns.regplot(
    data=student_df,
    x='mean_attendance_rate',
    y='mean_assessment_score',
    color='#7c3aed',
    scatter_kws={'alpha': 0.7, 's': 60},
    line_kws={'color': '#dc2626', 'linewidth': 2}
)

r_val = aggregator.calculate_attendance_score_correlation(student_df)
plt.title(f'Attendance Rate vs. Assessment Score (Pearson Correlation r = {r_val:.4f})')
plt.xlabel('Mean Attendance Rate (%)')
plt.ylabel('Mean Assessment Score (%)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 5. Summary Statistics Matrix & Export
Generate executive key statistics table (Mean, StdDev, Min, Max, Pass Rate).

In [ ]:
stats_summary = {
    'Metric': ['Assessment Score', 'Attendance Rate', 'Submission Rate', 'Academic Health Score'],
    'Mean': [
        student_df['mean_assessment_score'].mean(),
        student_df['mean_attendance_rate'].mean(),
        student_df['mean_submission_rate'].mean(),
        student_df['mean_health_score'].mean()
    ],
    'Std Dev': [
        student_df['mean_assessment_score'].std(),
        student_df['mean_attendance_rate'].std(),
        student_df['mean_submission_rate'].std(),
        student_df['mean_health_score'].std()
    ],
    'Min': [
        student_df['mean_assessment_score'].min(),
        student_df['mean_attendance_rate'].min(),
        student_df['mean_submission_rate'].min(),
        student_df['mean_health_score'].min()
    ],
    'Max': [
        student_df['mean_assessment_score'].max(),
        student_df['mean_attendance_rate'].max(),
        student_df['mean_submission_rate'].max(),
        student_df['mean_health_score'].max()
    ]
}

summary_table = pd.DataFrame(stats_summary).round(2)
display(summary_table)